In [1]:
import json, polars as pl
from pathlib import Path
from IPython.display import Image, Markdown, display
ROOT = Path.cwd().parent
T = ROOT / "reports" / "tables"; F = ROOT / "reports" / "figures"
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(200)
def show(p): display(Image(filename=str(p)))
def tab(p, cols=None):
    d = pl.read_csv(p); return d.select(cols) if cols else d

# 01 · Data quality
Source: Binance USDⓈ-M futures public archive, ETHUSDT perpetual, 2023-03-01 → 2026-09-30. The full report is generated by `scripts/run_stage2.py` (`docs/data_quality_report.md`). This notebook shows the key tables. Every value is read from the pipeline outputs.

In [2]:
display(Markdown((ROOT / 'docs' / 'data_quality_report.md').read_text().split('## 3.')[0]))

# Stage 2 — Data Acquisition & Data-Quality Report (ETHUSDT)

Generated by `scripts/run_stage2.py` → `ethof.quality.build_report`. Period **2023-03-01 → 2026-09-30** (UTC). Every figure below is computed from the pipeline outputs.

## 1. Raw files

Verified files (SHA-256 matches the published `.CHECKSUM`): **3,976**.

| dataset | status | n | GB |
|---|---|---|---|
| aggTrades | ok | 1,310 | 25.31 |
| bookDepth | ok | 1,310 | 0.6448 |
| bookTicker | ok | 3 | 0.4971 |
| fundingRate | ok | 43 | 3.828e-05 |
| metrics | ok | 1,310 | 0.01575 |

## 2. Trades (aggTrades)

- Raw aggregated trades read: **1,877,097,488** (from 1,310 daily files)
- Clean trades used for bars: **1,877,097,488**
- Timestamp unit detected per file: {'ms': 1310}
- Out-of-order rows in raw file order: time decreasing 0; agg id not increasing 0 (all data are sorted by (ts, agg_trade_id) anyway)
- Removed as data errors: **0** (null/non-finite 0; price ≤ 0 0; qty ≤ 0 0; first_id > last_id 0; outside file's day 0; bad side flag 0)
- Duplicates: exact rows 0; conflicting same-id rows 0
- agg_trade_id gaps within files: 0 (0 ids); across consecutive files: 0 boundaries with a gap
- Underlying trade-id gaps (first_trade_id ≠ previous last_trade_id + 1): 3,789,115 gaps, 4,047,288 ids missing = 0.2152% relative to agg-trade count; overlaps 0. Flagged, not repaired: the archive omits these fills from `aggTrades` (see §8).
- Bad-tick candidates (> 2% from the minute's median price): **333,456** trades in 69 bars (flagged, kept). 63 of these 69 bars are also in the extreme-move list (§3): the flags come from crash/squeeze minutes whose intra-minute range exceeds 2 %, not from isolated bad prints, so nothing is removed.



## Extreme one-minute moves (kept: liquidation cascades, not errors)

In [3]:
tab(ROOT / 'docs/stage2/extreme_1m_bars.csv', ['ts','open','high','low','close','n_trades','logret','classification']).head(15)

ts,open,high,low,close,n_trades,logret,classification
str,f64,f64,f64,f64,i64,f64,str
"""2023-03-12T17:26:00.000000+000…",1484.13,1535.0,1482.98,1514.29,36657,0.020111,"""sustained move (market event)"""
"""2023-03-24T11:47:00.000000+000…",1779.9,1781.12,1720.0,1738.93,27697,-0.023293,"""sustained move (market event)"""
"""2023-04-14T00:36:00.000000+000…",2026.46,2110.0,2025.09,2101.35,40794,0.03629,"""sustained move (market event)"""
"""2023-04-19T08:22:00.000000+000…",2013.97,2016.66,1948.48,1982.88,35355,-0.015528,"""sustained move (market event)"""
"""2023-08-17T21:41:00.000000+000…",1704.38,1704.54,1631.0,1655.7,16558,-0.028972,"""sustained move (market event)"""
"""2023-08-17T21:42:00.000000+000…",1645.33,1656.23,1600.0,1600.38,17569,-0.033983,"""sustained move (market event)"""
"""2023-08-17T21:44:00.000000+000…",1558.38,1559.91,1492.66,1498.65,18945,-0.039339,"""reverted, high activity"""
"""2023-08-17T21:45:00.000000+000…",1498.72,1581.8,1498.72,1571.69,18939,0.047587,"""sustained move (market event)"""
"""2023-10-01T22:17:00.000000+000…",1685.43,1780.0,1685.43,1768.77,24334,0.048258,"""sustained move (market event)"""


## Order-book coverage gaps (flagged, never interpolated)

In [4]:
tab(ROOT / 'docs/stage2/book_short_days.csv')

date,file_present,n_snapshots
str,bool,i64
"""2023-03-17""",true,354
"""2023-03-24""",true,2523
"""2023-06-01""",true,2027
"""2024-06-12""",true,2
"""2026-03-26""",true,2590
"""2026-04-09""",true,2575
"""2026-04-10""",true,2577
"""2026-04-16""",true,2576
"""2026-04-22""",true,2588


## Empirical check of the bookDepth sign convention and time zone

In [5]:
tab(ROOT / 'docs/stage2/book_sign_timezone_check.csv')

clock_shift_hours,n_matched,share_bid1_lt_vwap_lt_ask1,median_abs_bps_bandmid_vs_vwap
i64,i64,f64,f64
-8,3709804,0.294189,85.265678
-1,3710644,0.64209,27.671436
0,3710742,0.940561,3.144438
1,3710627,0.642991,27.862979
8,3709786,0.295136,84.514275


## Quoted spread (bookTicker sample days) used in the cost model

In [6]:
tab(T / 'spread_measurement.csv')

day,status,sha256,n_seconds,share_crossed_or_locked,share_one_tick,median_spread_usd,mean_spread_bps,median_spread_bps,p99_spread_bps
str,str,str,i64,f64,f64,f64,f64,f64,f64
"""2023-06-14""","""ok""","""3eec5c64cd960d6cdb4a41cffdb67c…",86392,0.0,0.994988,0.01,0.061831,0.057448,0.060843
"""2023-11-15""","""ok""","""5014091823ad7e34dea49ff59978f0…",86397,0.0,0.996007,0.01,0.050877,0.049787,0.050749
"""2024-03-13""","""ok""","""15754effd16b8dcee952b30b6cec14…",86400,0.0,0.9959375,0.01,0.025605,0.024924,0.025265
